# GAN Image Generation from an Uploaded Photograph

## Objective
Use a simple DCGAN-style architecture to demonstrate:

**Random Noise → Generator → Fake Image → Discriminator → Feedback → Weight Updates**

> Important: A GAN trained on one photograph (even with augmentation) is a teaching demonstration, not a reliable way to reconstruct the exact original photograph. For exact reconstruction, an autoencoder is more appropriate.

The notebook uses the uploaded image file `image(4).png` when available.

In [ ]:
# Install required libraries
!pip install tensorflow pillow matplotlib -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## 1. Load the uploaded photograph

The notebook first looks for the uploaded file in the current working directory and in `/mnt/data/`.

In [ ]:
# Locate the uploaded image
possible_paths = [
    "/mnt/data/image(4).png",
    "image(4).png",
    "/content/image(4).png"
]

IMAGE_PATH = next((p for p in possible_paths if os.path.exists(p)), None)

if IMAGE_PATH is None:
    raise FileNotFoundError(
        "Could not find image(4).png. Upload the image to the notebook environment."
    )

img = Image.open(IMAGE_PATH).convert("RGB")

print("Image path:", IMAGE_PATH)
print("Original image size:", img.size)

plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis("off")
plt.title("Original Uploaded Image")
plt.show()

## 2. Resize and normalize the image

For this simple experiment, images are represented as **64 × 64 × 3** RGB tensors.

Pixel values are normalized from **0–255** to **-1 to +1**, matching the `tanh` output of the Generator.

In [ ]:
IMG_SIZE = 64

img_resized = img.resize((IMG_SIZE, IMG_SIZE))

image_array = np.array(img_resized).astype("float32")
image_array = (image_array / 127.5) - 1.0

print("Resized image shape:", image_array.shape)
print("Pixel range:", image_array.min(), "to", image_array.max())

plt.figure(figsize=(5, 5))
plt.imshow((image_array + 1) / 2)
plt.axis("off")
plt.title("64 × 64 Normalized Training Image")
plt.show()

## 3. Create augmented real training images

We have only one photograph, so we create slightly different training examples using:

- horizontal flipping
- small rotation
- translation
- zoom

This gives the model a tiny image distribution to learn from.

In [ ]:
base_image = tf.convert_to_tensor(image_array, dtype=tf.float32)
base_image = tf.expand_dims(base_image, axis=0)

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.05)
])

NUM_IMAGES = 500
real_images = []

for _ in range(NUM_IMAGES):
    augmented = augmentation(base_image, training=True)
    real_images.append(augmented[0].numpy())

real_images = np.array(real_images)

print("Training dataset shape:", real_images.shape)

In [ ]:
# Visualize real augmented images
plt.figure(figsize=(10, 10))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    display_image = np.clip((real_images[i] + 1) / 2, 0, 1)
    plt.imshow(display_image)
    plt.axis("off")

plt.suptitle("Augmented Real Training Images", fontsize=16)
plt.tight_layout()
plt.show()

# 4. Build the Generator

The Generator converts a random latent vector into a 64 × 64 RGB image.

Architecture:

`100 random values → Dense → 4×4 → 8×8 → 16×16 → 32×32 → 64×64 RGB`

In [ ]:
LATENT_DIM = 100

generator = tf.keras.Sequential([
    layers.Input(shape=(LATENT_DIM,)),

    layers.Dense(4 * 4 * 512, use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(),
    layers.Reshape((4, 4, 512)),

    layers.Conv2DTranspose(
        256, 4, strides=2, padding="same", use_bias=False
    ),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(
        128, 4, strides=2, padding="same", use_bias=False
    ),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(
        64, 4, strides=2, padding="same", use_bias=False
    ),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    layers.Conv2DTranspose(
        3, 4, strides=2, padding="same", activation="tanh"
    )
])

generator.summary()

# 5. Build the Discriminator

The Discriminator receives a 64 × 64 RGB image and produces a score indicating whether it looks real or generated.

In [ ]:
discriminator = tf.keras.Sequential([
    layers.Input(shape=(64, 64, 3)),

    layers.Conv2D(64, 4, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(128, 4, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(256, 4, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(512, 4, strides=2, padding="same"),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(1)
])

discriminator.summary()

# 6. Check the Generator before training

Before learning, the Generator has random weights, so its output should look like noise or meaningless patterns.

In [ ]:
noise = tf.random.normal([1, LATENT_DIM])
before_training = generator(noise, training=False)[0]

plt.figure(figsize=(5, 5))
plt.imshow(np.clip((before_training + 1) / 2, 0, 1))
plt.axis("off")
plt.title("Generator Output Before Training")
plt.show()

# 7. Define GAN losses

Discriminator goal:

- Real image → 1
- Fake image → 0

Generator goal:

- Fake image should be classified as real → 1

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(
        tf.ones_like(real_output),
        real_output
    )
    fake_loss = cross_entropy(
        tf.zeros_like(fake_output),
        fake_output
    )
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

# 8. One GAN training step

Each training step:

1. Generate fake images from random noise.
2. Ask the Discriminator to classify real images.
3. Ask the Discriminator to classify fake images.
4. Calculate Generator and Discriminator losses.
5. Calculate gradients.
6. Update both networks.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal(
        [tf.shape(images)[0], LATENT_DIM]
    )

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:

        generated_images = generator(
            noise,
            training=True
        )

        real_output = discriminator(
            images,
            training=True
        )

        fake_output = discriminator(
            generated_images,
            training=True
        )

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(
            real_output,
            fake_output
        )

    gen_gradients = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    disc_gradients = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(
            gen_gradients,
            generator.trainable_variables
        )
    )

    discriminator_optimizer.apply_gradients(
        zip(
            disc_gradients,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss

# 9. Prepare the TensorFlow dataset

In [ ]:
BATCH_SIZE = 32

dataset = (
    tf.data.Dataset
    .from_tensor_slices(real_images)
    .shuffle(len(real_images))
    .batch(BATCH_SIZE)
)

print("Number of batches:", tf.data.experimental.cardinality(dataset).numpy())

# 10. Train and observe Generator learning

A fixed set of random noise vectors is used for visualization. Because the input stays the same, students can compare the Generator's output across epochs.

In [ ]:
EPOCHS = 300
DISPLAY_EVERY = 25

seed = tf.random.normal([16, LATENT_DIM])

history_g = []
history_d = []
snapshots = {}

for epoch in range(EPOCHS + 1):

    if epoch > 0:
        epoch_g = []
        epoch_d = []

        for batch in dataset:
            g_loss, d_loss = train_step(batch)
            epoch_g.append(float(g_loss))
            epoch_d.append(float(d_loss))

        mean_g = np.mean(epoch_g)
        mean_d = np.mean(epoch_d)

        history_g.append(mean_g)
        history_d.append(mean_d)

    if epoch % DISPLAY_EVERY == 0:
        generated = generator(seed, training=False).numpy()
        snapshots[epoch] = generated

        print(
            f"Epoch {epoch:03d} | "
            f"Generator Loss: {history_g[-1]:.4f} | "
            f"Discriminator Loss: {history_d[-1]:.4f}"
            if history_g else
            f"Epoch {epoch:03d} | Initial output"
        )

# 11. Visualize Generator learning across training

Compare the same latent inputs at different training stages.

In [ ]:
selected_epochs = [0, 25, 50, 100, 200, 300]

fig = plt.figure(figsize=(15, 15))

for row, epoch in enumerate(selected_epochs):

    generated = snapshots[epoch]

    for col in range(4):

        index = row * 4 + col

        ax = plt.subplot(
            len(selected_epochs),
            4,
            index + 1
        )

        image = np.clip(
            (generated[col] + 1) / 2,
            0,
            1
        )

        ax.imshow(image)
        ax.axis("off")

        if col == 0:
            ax.set_ylabel(
                f"Epoch {epoch}",
                fontsize=12
            )

plt.suptitle(
    "Generator Learning Progress",
    fontsize=18
)

plt.tight_layout()
plt.show()

# 12. Plot Generator and Discriminator losses

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(history_g) + 1),
    history_g,
    label="Generator Loss"
)

plt.plot(
    range(1, len(history_d) + 1),
    history_d,
    label="Discriminator Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.grid(True)
plt.show()

# 13. Final generated images

These images are generated from random noise after training.

In [ ]:
final_images = generator(
    tf.random.normal([16, LATENT_DIM]),
    training=False
).numpy()

plt.figure(figsize=(10, 10))

for i in range(16):

    plt.subplot(4, 4, i + 1)

    image = np.clip(
        (final_images[i] + 1) / 2,
        0,
        1
    )

    plt.imshow(image)
    plt.axis("off")

plt.suptitle(
    "Final GAN Generated Images",
    fontsize=16
)

plt.tight_layout()
plt.show()

# 14. Test the Discriminator

The Discriminator should assign different scores to real and generated images.

Because the Discriminator uses logits, larger positive values indicate stronger belief that an image is real.

In [ ]:
real_batch = real_images[:8]

fake_batch = generator(
    tf.random.normal([8, LATENT_DIM]),
    training=False
)

real_scores = discriminator(
    real_batch,
    training=False
).numpy().flatten()

fake_scores = discriminator(
    fake_batch,
    training=False
).numpy().flatten()

print("Real image discriminator scores:")
print(np.round(real_scores, 3))

print("\nFake image discriminator scores:")
print(np.round(fake_scores, 3))

# 15. Key learning points

### Generator
Learns to transform random noise into images that resemble the training distribution.

### Discriminator
Learns to distinguish real training images from generated images.

### Adversarial learning
The Generator tries to fool the Discriminator, while the Discriminator tries not to be fooled.

### Important limitation
Because this experiment uses only one original photograph plus augmentation, it is **not a proper face-generation dataset**. The output may contain artifacts, blur, or memorization. A real GAN image-generation experiment should use many training images.

### Exact reconstruction
If the objective is:

`Uploaded photograph → reconstructed version of the same photograph`

use an **Autoencoder** rather than a GAN.